# 🔬 RAG Retrieval Comparison
## BM25 vs DPR vs Hybrid — نتائج حقيقية

**قبل تشغيل أي شي:**
1. فعّل GPU: `Runtime → Change runtime type → T4 GPU`
2. شغّل الخلايا بالترتيب من فوق لتحت

---

## الخطوة 1 — تثبيت المكتبات

In [ ]:
# ⏳ هاد بياخد 2-3 دقائق — انتظر لحين ما تشوف ✅
!pip install rank-bm25==0.2.2 sentence-transformers faiss-gpu datasets transformers -q
print('✅ تم تثبيت المكتبات')

## الخطوة 2 — Corpus و Queries (100 passage + 30 query)

In [ ]:
PASSAGES = [
    {"id":"p00","text":"Albert Einstein was a German-born theoretical physicist who developed the theory of relativity. Born on March 14, 1879 in Ulm, Germany."},
    {"id":"p01","text":"Einstein's special theory of relativity published in 1905 introduced E=mc². General relativity in 1915 described gravity as curvature of spacetime."},
    {"id":"p02","text":"The Nobel Prize in Physics 1921 was awarded to Albert Einstein for his discovery of the law of the photoelectric effect."},
    {"id":"p03","text":"Isaac Newton formulated the laws of motion and universal gravitation in his 1687 work Principia Mathematica."},
    {"id":"p04","text":"Quantum mechanics describes nature at atomic and subatomic scales, developed by Niels Bohr, Werner Heisenberg, and Erwin Schrödinger."},
    {"id":"p05","text":"Black holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape."},
    {"id":"p06","text":"The first image of a black hole was captured in 2019 by the Event Horizon Telescope showing M87* located 55 million light-years away."},
    {"id":"p07","text":"Charles Darwin proposed evolution by natural selection in his 1859 book On the Origin of Species."},
    {"id":"p08","text":"DNA double-helix structure was discovered by James Watson and Francis Crick in 1953 using X-ray crystallography data from Rosalind Franklin."},
    {"id":"p09","text":"The Human Genome Project completed in 2003 produced the first complete sequence of the human genome containing 3 billion base pairs."},
    {"id":"p10","text":"Penicillin was discovered by Alexander Fleming in 1928 when he noticed the mold Penicillium notatum was killing surrounding bacteria."},
    {"id":"p11","text":"Howard Florey and Ernst Boris Chain developed penicillin into a usable antibiotic. Fleming, Florey, and Chain shared the 1945 Nobel Prize."},
    {"id":"p12","text":"Marie Curie was the first woman to win a Nobel Prize and the only person to win in two sciences: Physics 1903 and Chemistry 1911."},
    {"id":"p13","text":"The World Wide Web was invented by Tim Berners-Lee in 1989 while working at CERN in Switzerland."},
    {"id":"p14","text":"Tim Berners-Lee wrote the first web browser WorldWideWeb in 1990. The web became publicly available in 1991."},
    {"id":"p15","text":"Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability and is widely used in AI."},
    {"id":"p16","text":"Deep learning uses neural networks with many layers achieving breakthroughs in image recognition and natural language processing."},
    {"id":"p17","text":"The International Space Station has been continuously inhabited since November 2000. It orbits at 408 km altitude, 15.5 orbits per day."},
    {"id":"p18","text":"The Berlin Wall fell on November 9, 1989 after East Germany announced citizens could cross the border freely."},
    {"id":"p19","text":"German reunification was completed October 3, 1990, merging East and West Germany into a single democratic state."},
    {"id":"p20","text":"The Titanic sank on April 15, 1912 after striking an iceberg. Approximately 1,517 of 2,224 passengers and crew died."},
    {"id":"p21","text":"The Titanic was designed by Thomas Andrews and built by Harland and Wolff in Belfast. It was owned by the White Star Line."},
    {"id":"p22","text":"Mount Everest stands at 8,848.86 metres above sea level on the Nepal-Tibet border in the Himalayas."},
    {"id":"p23","text":"The first ascent of Everest was May 29, 1953 by Edmund Hillary of New Zealand and Tenzing Norgay, a Nepali Sherpa."},
    {"id":"p24","text":"The Mona Lisa was painted by Leonardo da Vinci between 1503 and 1519. It is kept at the Louvre Museum in Paris."},
    {"id":"p25","text":"Leonardo da Vinci was born April 15, 1452 in Vinci, Florence. He is considered the greatest painter and polymath who ever lived."},
    {"id":"p26","text":"Othello is a tragedy by William Shakespeare written around 1603. The villain Iago manipulates general Othello into jealousy."},
    {"id":"p27","text":"William Shakespeare was born in April 1564 in Stratford-upon-Avon. He wrote 37 plays and 154 sonnets."},
    {"id":"p28","text":"The Eiffel Tower was designed by Gustave Eiffel for the 1889 World Fair in Paris. It stands 330 metres tall."},
    {"id":"p29","text":"Apollo 11 launched July 16, 1969. Neil Armstrong was the first human to walk on the Moon on July 20, 1969."},
    {"id":"p30","text":"Mars is called the Red Planet due to iron oxide on its surface. NASA Perseverance rover landed February 18, 2021."},
    {"id":"p31","text":"Perseverance rover landed in Jezero Crater on Mars to search for ancient microbial life and collect rock samples."},
    {"id":"p32","text":"The Paris Agreement adopted December 2015 aims to limit global warming to 1.5 degrees Celsius above pre-industrial levels."},
    {"id":"p33","text":"Climate change since the 19th century is driven by burning fossil fuels releasing greenhouse gases into the atmosphere."},
    {"id":"p34","text":"Photosynthesis converts sunlight, CO2 and water into glucose and oxygen: 6CO2 + 6H2O + light → C6H12O6 + 6O2."},
    {"id":"p35","text":"CRISPR-Cas9 allows precise gene editing in living organisms. Doudna and Charpentier won the 2020 Nobel Prize in Chemistry."},
    {"id":"p36","text":"Voyager 1 launched 1977 became the most distant human-made object. In 2012 it entered interstellar space."},
    {"id":"p37","text":"The Great Depression began with the stock market crash October 29, 1929 (Black Tuesday) and lasted through the 1930s."},
    {"id":"p38","text":"COVID-19 pandemic caused by SARS-CoV-2 was declared a global pandemic by WHO on March 11, 2020."},
    {"id":"p39","text":"mRNA vaccines for COVID-19 by Pfizer-BioNTech and Moderna received emergency authorization in December 2020."},
    {"id":"p40","text":"The Hubble Space Telescope launched April 24, 1990 and has operated over 30 years contributing to major space discoveries."},
    # Multi-hop bridge passages
    {"id":"p41","text":"Christopher Nolan was born July 30, 1970 in London, holding British and American citizenship."},
    {"id":"p42","text":"Inception (2010) is a science fiction film written and directed by Christopher Nolan starring Leonardo DiCaprio."},
    {"id":"p43","text":"Oppenheimer (2023) is a biographical film written and directed by Christopher Nolan about J. Robert Oppenheimer."},
    {"id":"p44","text":"J. Robert Oppenheimer led the Manhattan Project and is known as the father of the atomic bomb."},
    {"id":"p45","text":"The Louvre Museum in Paris is the world's largest art museum attracting 9.6 million visitors per year."},
    {"id":"p46","text":"Leonardo da Vinci painted The Last Supper 1495-1498, a mural in Milan depicting Jesus and his apostles."},
    {"id":"p47","text":"Alexander Fleming was born August 6, 1881 in Lochfield, Scotland. He studied at St Mary's Hospital, London."},
    {"id":"p48","text":"St Mary's Hospital is in Paddington, London where Fleming discovered penicillin. It now has the Fleming Laboratory Museum."},
    {"id":"p49","text":"Apple Inc. was co-founded by Steve Jobs, Steve Wozniak, and Ronald Wayne on April 1, 1976."},
    {"id":"p50","text":"The first iPhone was announced by Steve Jobs January 9, 2007 and went on sale June 29, 2007."},
]

QUERIES = [
    # FACTOID
    {"id":"q00","type":"factoid","question":"Who discovered penicillin and in what year?","relevant_ids":["p10","p11","p47"],"answer":"Alexander Fleming, 1928"},
    {"id":"q01","type":"factoid","question":"When did the Berlin Wall fall?","relevant_ids":["p18","p19"],"answer":"November 9, 1989"},
    {"id":"q02","type":"factoid","question":"Who invented the World Wide Web?","relevant_ids":["p13","p14"],"answer":"Tim Berners-Lee, 1989"},
    {"id":"q03","type":"factoid","question":"What is Einstein's famous mass-energy equation?","relevant_ids":["p01","p00"],"answer":"E=mc2"},
    {"id":"q04","type":"factoid","question":"When was the first iPhone released?","relevant_ids":["p50","p49"],"answer":"June 29, 2007"},
    {"id":"q05","type":"factoid","question":"Who was the first human to walk on the Moon?","relevant_ids":["p29"],"answer":"Neil Armstrong, July 20 1969"},
    {"id":"q06","type":"factoid","question":"Which Shakespeare play features the character Iago?","relevant_ids":["p26","p27"],"answer":"Othello"},
    {"id":"q07","type":"factoid","question":"Where is the Mona Lisa kept?","relevant_ids":["p24","p45"],"answer":"The Louvre Museum Paris"},
    {"id":"q08","type":"factoid","question":"How many people died when the Titanic sank?","relevant_ids":["p20","p21"],"answer":"About 1517 people"},
    {"id":"q09","type":"factoid","question":"What year was COVID-19 declared a pandemic?","relevant_ids":["p38","p39"],"answer":"2020"},
    # SEMANTIC
    {"id":"q10","type":"semantic","question":"Which physicist explained why objects fall using mathematical laws?","relevant_ids":["p03"],"answer":"Isaac Newton"},
    {"id":"q11","type":"semantic","question":"What technology allows scientists to precisely edit genes?","relevant_ids":["p35"],"answer":"CRISPR-Cas9"},
    {"id":"q12","type":"semantic","question":"Which international agreement limits temperature rise from fossil fuels?","relevant_ids":["p32","p33"],"answer":"Paris Agreement"},
    {"id":"q13","type":"semantic","question":"Which spacecraft traveled farthest from our solar system?","relevant_ids":["p36"],"answer":"Voyager 1"},
    {"id":"q14","type":"semantic","question":"What process do plants use to convert sunlight into food?","relevant_ids":["p34"],"answer":"Photosynthesis"},
    {"id":"q15","type":"semantic","question":"What severe economic crisis followed the 1929 financial crash?","relevant_ids":["p37"],"answer":"The Great Depression"},
    {"id":"q16","type":"semantic","question":"What satellite has orbited Earth for over three decades capturing space images?","relevant_ids":["p40"],"answer":"Hubble Space Telescope"},
    {"id":"q17","type":"semantic","question":"What molecule carries the hereditary blueprint of living organisms?","relevant_ids":["p08","p09"],"answer":"DNA"},
    {"id":"q18","type":"semantic","question":"Who received a Nobel Prize for developing the technology behind antibiotics?","relevant_ids":["p10","p11"],"answer":"Fleming Florey Chain 1945"},
    {"id":"q19","type":"semantic","question":"What rover landed on the Red Planet in 2021?","relevant_ids":["p30","p31"],"answer":"Perseverance"},
    # MULTI-HOP
    {"id":"q20","type":"multi-hop","question":"What is the nationality of the director of the film Inception?","relevant_ids":["p42","p41"],"answer":"British-American"},
    {"id":"q21","type":"multi-hop","question":"Who directed the film about the physicist who led the Manhattan Project?","relevant_ids":["p43","p44"],"answer":"Christopher Nolan"},
    {"id":"q22","type":"multi-hop","question":"In which museum is the most famous painting by the artist who painted The Last Supper?","relevant_ids":["p24","p46","p45"],"answer":"The Louvre Museum Paris"},
    {"id":"q23","type":"multi-hop","question":"In which London area was the hospital where the discoverer of penicillin worked?","relevant_ids":["p47","p48","p10"],"answer":"Paddington London"},
    {"id":"q24","type":"multi-hop","question":"Who co-founded the company that produced the first iPhone?","relevant_ids":["p50","p49"],"answer":"Steve Jobs Apple"},
    {"id":"q25","type":"multi-hop","question":"Which country built the ship that sank killing over 1500 people in 1912?","relevant_ids":["p20","p21"],"answer":"United Kingdom Belfast"},
    {"id":"q26","type":"multi-hop","question":"What rover landed on the planet with iron oxide making it appear red?","relevant_ids":["p30","p31"],"answer":"Perseverance on Mars"},
    {"id":"q27","type":"multi-hop","question":"What is the birthplace city of the director of Oppenheimer?","relevant_ids":["p43","p41"],"answer":"London"},
    {"id":"q28","type":"multi-hop","question":"Who painted the artwork kept in the world largest art museum in Paris?","relevant_ids":["p24","p45","p25"],"answer":"Leonardo da Vinci Mona Lisa"},
    {"id":"q29","type":"multi-hop","question":"In what year was the co-founder of the company that made the first touchscreen smartphone born?","relevant_ids":["p49","p50"],"answer":"Steve Jobs 1955"},
]

print(f'✅ Corpus: {len(PASSAGES)} passages')
print(f'✅ Queries: {len(QUERIES)} queries')
types = {}
for q in QUERIES:
    types[q['type']] = types.get(q['type'], 0) + 1
print(f'   Types: {types}')

## الخطوة 3 — BM25 Retriever

In [ ]:
import time, string
from rank_bm25 import BM25Okapi

def tokenize(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

class BM25Retriever:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.corpus = corpus
        self.ids    = [p['id']   for p in corpus]
        self.texts  = [p['text'] for p in corpus]
        self.bm25   = None

    def build_index(self):
        t0 = time.perf_counter()
        self.bm25 = BM25Okapi([tokenize(t) for t in self.texts])
        ms = (time.perf_counter()-t0)*1000
        print(f'[BM25] ✅ Index built — {len(self.corpus)} passages | {ms:.0f} ms')

    def retrieve(self, query, top_k=5):
        t0     = time.perf_counter()
        scores = self.bm25.get_scores(tokenize(query))
        ms     = (time.perf_counter()-t0)*1000
        ranked = sorted(zip(self.ids, self.texts, scores),
                        key=lambda x: x[2], reverse=True)[:top_k]
        return [{'id':pid,'text':txt,'score':float(sc),'rank':i+1,'latency_ms':ms}
                for i,(pid,txt,sc) in enumerate(ranked)]

bm25 = BM25Retriever(PASSAGES)
bm25.build_index()

# اختبار سريع
r = bm25.retrieve('Who discovered penicillin?', top_k=3)
print('\nTest query: Who discovered penicillin?')
for x in r:
    print(f"  #{x['rank']} [{x['id']}] score={x['score']:.3f} | {x['text'][:70]}...")

## الخطوة 4 — DPR Retriever (بيحمّل الـ model تلقائياً)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

class DPRRetriever:
    def __init__(self, corpus, model_name='multi-qa-mpnet-base-dot-v1'):
        self.corpus = corpus
        self.ids    = [p['id']   for p in corpus]
        self.texts  = [p['text'] for p in corpus]
        self.model  = None
        self.index  = None
        self.model_name = model_name

    def build_index(self):
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'[DPR] Loading model on {device}...')
        self.model = SentenceTransformer(self.model_name, device=device)

        print(f'[DPR] Encoding {len(self.corpus)} passages...')
        t0 = time.perf_counter()
        embs = self.model.encode(self.texts, batch_size=64,
                                  show_progress_bar=True,
                                  normalize_embeddings=True).astype('float32')
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)
        ms = (time.perf_counter()-t0)*1000
        print(f'[DPR] ✅ Index built — {self.index.ntotal} vectors | {ms:.0f} ms')

    def retrieve(self, query, top_k=5):
        t0    = time.perf_counter()
        q_vec = self.model.encode([query], normalize_embeddings=True).astype('float32')
        scores, idxs = self.index.search(q_vec, top_k)
        ms = (time.perf_counter()-t0)*1000
        return [{'id':self.ids[i],'text':self.texts[i],
                 'score':float(s),'rank':r+1,'latency_ms':ms}
                for r,(i,s) in enumerate(zip(idxs[0], scores[0])) if i>=0]

dpr = DPRRetriever(PASSAGES)
dpr.build_index()

# اختبار
r = dpr.retrieve('Who discovered penicillin?', top_k=3)
print('\nTest query: Who discovered penicillin?')
for x in r:
    print(f"  #{x['rank']} [{x['id']}] score={x['score']:.4f} | {x['text'][:70]}...")

## الخطوة 5 — Hybrid Retriever (RRF + Alpha)

In [ ]:
class HybridRetriever:
    def __init__(self, bm25, dpr, fusion='rrf', rrf_k=60, alpha=0.4, factor=3):
        self.bm25   = bm25
        self.dpr    = dpr
        self.fusion = fusion
        self.rrf_k  = rrf_k
        self.alpha  = alpha
        self.factor = factor

    def retrieve(self, query, top_k=5):
        k = top_k * self.factor
        t0 = time.perf_counter()
        b  = self.bm25.retrieve(query, k)
        d  = self.dpr.retrieve(query, k)
        ms = (time.perf_counter()-t0)*1000

        if self.fusion == 'rrf':
            scores, texts = {}, {}
            for r in b: scores[r['id']] = scores.get(r['id'],0) + 1/(self.rrf_k+r['rank']); texts[r['id']]=r['text']
            for r in d: scores[r['id']] = scores.get(r['id'],0) + 1/(self.rrf_k+r['rank']); texts[r['id']]=r['text']
            bm25_map = {r['id']:r['rank'] for r in b}
            dpr_map  = {r['id']:r['rank'] for r in d}
            merged = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
            return [{'id':pid,'text':texts[pid],'score':sc,'rank':i+1,
                     'bm25_rank':bm25_map.get(pid),'dpr_rank':dpr_map.get(pid),
                     'latency_ms':ms} for i,(pid,sc) in enumerate(merged)]
        else:  # alpha
            def norm(res):
                s = [r['score'] for r in res]
                lo,hi = min(s),max(s)
                d = hi-lo if hi!=lo else 1
                return {r['id']:(r['score']-lo)/d for r in res}
            bn,dn = norm(b),norm(d)
            texts = {r['id']:r['text'] for r in b+d}
            comb  = {pid: self.alpha*bn.get(pid,0)+(1-self.alpha)*dn.get(pid,0)
                     for pid in set(bn)|set(dn)}
            merged = sorted(comb.items(), key=lambda x:x[1], reverse=True)[:top_k]
            return [{'id':pid,'text':texts[pid],'score':sc,'rank':i+1,'latency_ms':ms}
                    for i,(pid,sc) in enumerate(merged)]

hybrid_rrf   = HybridRetriever(bm25, dpr, fusion='rrf',   rrf_k=60)
hybrid_alpha = HybridRetriever(bm25, dpr, fusion='alpha',  alpha=0.4)

r = hybrid_rrf.retrieve('Who discovered penicillin?', top_k=3)
print('Test — Hybrid RRF: Who discovered penicillin?')
for x in r:
    b_r = x.get('bm25_rank','-'); d_r = x.get('dpr_rank','-')
    print(f"  #{x['rank']} [{x['id']}] rrf={x['score']:.4f} bm25#{b_r} dpr#{d_r} | {x['text'][:60]}...")
print('✅ Hybrid ready')

## الخطوة 6 — Metrics (كل القياسات)

In [ ]:
import re, string
from collections import Counter

def normalize(text):
    text = text.lower().translate(str.maketrans('','',string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    if not relevant: return 0.0
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def mrr(retrieved, relevant, k=10):
    for i, pid in enumerate(retrieved[:k], 1):
        if pid in set(relevant): return 1.0/i
    return 0.0

def hit_at_k(retrieved, relevant, k):
    return float(bool(set(retrieved[:k]) & set(relevant)))

def token_f1(answer, context):
    a = Counter(normalize(answer).split())
    c = Counter(normalize(context).split())
    common = sum((a & c).values())
    if not common: return 0.0
    p = common / sum(c.values())
    r = common / sum(a.values())
    return 2*p*r/(p+r)

def exact_match(answer, context):
    return float(normalize(answer) in normalize(context))

def faithfulness(answer, context):
    a = set(normalize(answer).split())
    c = set(normalize(context).split())
    return len(a & c) / len(a) if a else 0.0

def answer_relevance(answer, question):
    a = set(normalize(answer).split())
    q = set(normalize(question).split())
    return len(a & q) / max(len(a), len(q)) if a and q else 0.0

print('✅ Metrics functions ready')
# test
print(f'  P@5  test: {precision_at_k(["p10","p11","p00","p01","p02"],["p10","p11"],5):.2f}')
print(f'  MRR  test: {mrr(["p00","p10","p11"],["p10","p11"]):.2f}')
print(f'  F1   test: {token_f1("Alexander Fleming 1928","Penicillin was discovered by Alexander Fleming in 1928"):.2f}')

## الخطوة 7 — تشغيل التجارب وعرض النتائج

In [ ]:
from tqdm.notebook import tqdm

def evaluate(name, retriever, queries, top_k=5):
    P, R, M, F, E, Faith, Rel, Lat = [], [], [], [], [], [], [], []
    for q in tqdm(queries, desc=f'{name}'):
        res          = retriever.retrieve(q['question'], top_k)
        retrieved_ids = [r['id'] for r in res]
        relevant_ids  = q['relevant_ids']
        ctx           = res[0]['text'] if res else ''
        ans           = q.get('answer', '')

        P.append(precision_at_k(retrieved_ids, relevant_ids, top_k))
        R.append(recall_at_k(   retrieved_ids, relevant_ids, top_k))
        M.append(mrr(           retrieved_ids, relevant_ids))
        F.append(token_f1(ans, ctx))
        E.append(exact_match(ans, ctx))
        Faith.append(faithfulness(ans, ctx))
        Rel.append(answer_relevance(ans, q['question']))
        Lat.append(res[0]['latency_ms'] if res else 0)

    return {
        'Method'       : name,
        'P@5'          : round(np.mean(P),   4),
        'R@5'          : round(np.mean(R),   4),
        'MRR'          : round(np.mean(M),   4),
        'F1'           : round(np.mean(F),   4),
        'EM'           : round(np.mean(E),   4),
        'Faithfulness' : round(np.mean(Faith),4),
        'Ans.Relevance': round(np.mean(Rel), 4),
        'Latency(ms)'  : round(np.mean(Lat), 2),
    }

TOP_K = 5
print(f'🔬 Running experiments — top_k={TOP_K}, queries={len(QUERIES)}\n')

r_bm25  = evaluate('BM25',         bm25,        QUERIES, TOP_K)
r_dpr   = evaluate('DPR',          dpr,         QUERIES, TOP_K)
r_rrf   = evaluate('Hybrid-RRF',   hybrid_rrf,  QUERIES, TOP_K)
r_alpha = evaluate('Hybrid-α=0.4', hybrid_alpha,QUERIES, TOP_K)

results = [r_bm25, r_dpr, r_rrf, r_alpha]
print('\n✅ Done!')

## الخطوة 8 — جدول النتائج الكامل

In [ ]:
import pandas as pd

df = pd.DataFrame(results).set_index('Method')

def highlight_best(s):
    if s.name == 'Latency(ms)':
        best = s.min()
    else:
        best = s.max()
    return ['background-color: #d4efdf; font-weight: bold' if v == best else '' for v in s]

print('='*70)
print('  MAIN RESULTS TABLE — All Methods (green = best)')
print('='*70)
display(df.style.apply(highlight_best))

print('\n📊 Best method per metric:')
for col in df.columns:
    if col == 'Latency(ms)':
        best = df[col].idxmin()
    else:
        best = df[col].idxmax()
    print(f'  {col:18s}: {best} ({df.loc[best,col]})')

## الخطوة 9 — تحليل حسب نوع السؤال

In [ ]:
print('Per Query-Type Analysis (MRR)\n' + '='*50)

type_rows = []
for qtype in ['factoid', 'semantic', 'multi-hop']:
    typed = [q for q in QUERIES if q['type'] == qtype]
    row = {'Query Type': f'{qtype} (n={len(typed)})'}
    for name, retriever in [('BM25', bm25),('DPR', dpr),
                             ('Hybrid-RRF', hybrid_rrf),('Hybrid-α', hybrid_alpha)]:
        mrrs = []
        for q in typed:
            res = retriever.retrieve(q['question'], TOP_K)
            mrrs.append(mrr([r['id'] for r in res], q['relevant_ids']))
        row[name] = round(np.mean(mrrs), 4)
    type_rows.append(row)

df_types = pd.DataFrame(type_rows).set_index('Query Type')
display(df_types.style.apply(highlight_best))

print('\n💡 Key observation:')
for i, row in df_types.iterrows():
    best = row.idxmax()
    worst = row.idxmin()
    gap = row.max() - row.min()
    print(f'  {i}: Best={best} | Gap={gap:.4f} (best vs worst)')

## الخطوة 10 — Ablation Study: Alpha Sweep

In [ ]:
print('Ablation — Alpha sweep (BM25 weight α)\n')
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
alpha_rows = []

for a in tqdm(alphas, desc='Alpha sweep'):
    h = HybridRetriever(bm25, dpr, fusion='alpha', alpha=a)
    mrrs, precs, recs = [], [], []
    for q in QUERIES:
        res = h.retrieve(q['question'], TOP_K)
        ids = [r['id'] for r in res]
        mrrs.append( mrr(           ids, q['relevant_ids']))
        precs.append(precision_at_k(ids, q['relevant_ids'], TOP_K))
        recs.append( recall_at_k(   ids, q['relevant_ids'], TOP_K))
    alpha_rows.append({'α': a,
                       'MRR':  round(np.mean(mrrs),  4),
                       'P@5':  round(np.mean(precs), 4),
                       'R@5':  round(np.mean(recs),  4)})

df_alpha = pd.DataFrame(alpha_rows).set_index('α')
display(df_alpha.style.apply(highlight_best))

best_a = df_alpha['MRR'].idxmax()
print(f'\n✅ Best α = {best_a} → MRR = {df_alpha.loc[best_a,"MRR"]}')
print('   (α=0 = pure DPR, α=1 = pure BM25)')

## الخطوة 11 — Ablation Study: RRF-k Sweep

In [ ]:
print('Ablation — RRF-k sweep\n')
k_values = [10, 20, 40, 60, 80, 100]
k_rows   = []

for k_val in tqdm(k_values, desc='RRF-k sweep'):
    h = HybridRetriever(bm25, dpr, fusion='rrf', rrf_k=k_val)
    mrrs, precs = [], []
    for q in QUERIES:
        res = h.retrieve(q['question'], TOP_K)
        ids = [r['id'] for r in res]
        mrrs.append( mrr(           ids, q['relevant_ids']))
        precs.append(precision_at_k(ids, q['relevant_ids'], TOP_K))
    k_rows.append({'RRF-k': k_val,
                   'MRR': round(np.mean(mrrs), 4),
                   'P@5': round(np.mean(precs),4)})

df_k = pd.DataFrame(k_rows).set_index('RRF-k')
display(df_k.style.apply(highlight_best))

best_k = df_k['MRR'].idxmax()
print(f'\n✅ Best k = {best_k} → MRR = {df_k.loc[best_k,"MRR"]}')

## الخطوة 12 — حفظ النتائج وتحميلها

In [ ]:
import json, os
from google.colab import files

os.makedirs('results', exist_ok=True)

# حفظ CSV
df.to_csv('results/main_results.csv')
df_types.to_csv('results/per_type_results.csv')
df_alpha.to_csv('results/ablation_alpha.csv')
df_k.to_csv('results/ablation_rrf_k.csv')

# حفظ JSON
all_results = {
    'main'        : results,
    'per_type'    : df_types.reset_index().to_dict('records'),
    'alpha_sweep' : alpha_rows,
    'rrf_k_sweep' : k_rows,
    'best_alpha'  : float(best_a),
    'best_rrf_k'  : int(best_k),
}
with open('results/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('✅ Files saved:')
for f in os.listdir('results'):
    print(f'   results/{f}')

# تحميل مباشر
print('\n📥 Downloading results...')
files.download('results/main_results.csv')
files.download('results/ablation_alpha.csv')
files.download('results/all_results.json')

## ✅ انتهى! — ملخص النتائج

In [ ]:
print('='*60)
print('  FINAL SUMMARY')
print('='*60)
print(f'\n  Dataset : {len(PASSAGES)} passages, {len(QUERIES)} queries')
print(f'  top_k   : {TOP_K}')
print(f'  Best α  : {best_a}')
print(f'  Best k  : {best_k}\n')

print('  Main results:')
for row in results:
    print(f"  {row['Method']:18s} MRR={row['MRR']:.4f}  F1={row['F1']:.4f}  Faith={row['Faithfulness']:.4f}  Lat={row['Latency(ms)']:.1f}ms")

print('\n  انسخ هالأرقام مباشرة على جداول الورقة البحثية ✅')